# 1D CNN Autoencoder — NIDS Training

Train multiple 1D-CNN autoencoder configurations on Monday benign traffic.

**Key idea:** Conv1D treats the 78 features as a 1-D signal, capturing local
feature correlations. Good at detecting anomalous patterns in feature neighbourhoods.

In [1]:
import sys, os

# Mount Google Drive and add the project directory to Python path
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from tensorflow.keras import layers, callbacks

import joblib
import nids_utils as nu

print(f"TF version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TF version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load & Preprocess Data

In [2]:
SCALER_SAVE = os.path.join(nu.PREPROCESSING_DIR, "scaler_cnnae.pkl")

X_train, scaler, feature_cols = nu.prepare_train_data(
    monday_path=nu.MONDAY_CSV,
    scaler_save_path=SCALER_SAVE,
)

input_dim = X_train.shape[1]
print(f"Training samples: {X_train.shape[0]}, features: {input_dim}")

# Reshape to (samples, timesteps, 1) for Conv1D
X_train_cnn = X_train.reshape(-1, input_dim, 1)
print(f"CNN input shape: {X_train_cnn.shape}")

Loaded /content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Monday-WorkingHours.pcap_ISCX.csv  shape=(529918, 79)
Label distribution:
Label
BENIGN    529918
Name: count, dtype: int64
Scaler saved → /content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/scaler_cnnae.pkl
Training samples: 529918, features: 78
CNN input shape: (529918, 78, 1)


## 2. CNN-AE Model Definition

In [3]:
@keras.saving.register_keras_serializable()
class CNNEncoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters

        self.conv_layers = []
        for f in filters:
            self.conv_layers.append(
                layers.Conv1D(f, kernel_size, padding="same", activation="relu")
            )
        self.bottleneck = layers.Conv1D(
            latent_filters, kernel_size, padding="same", activation="relu"
        )
        self.pool = layers.MaxPooling1D(2, padding="same")

    def call(self, x):
        for conv in self.conv_layers:
            x = conv(x)
        x = self.bottleneck(x)
        return self.pool(x)

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }


@keras.saving.register_keras_serializable()
class CNNDecoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters

        self.deconv_entry = layers.Conv1D(
            latent_filters, kernel_size, padding="same", activation="relu"
        )
        self.upsample = layers.UpSampling1D(2)
        self.conv_layers = []
        for f in reversed(filters):
            self.conv_layers.append(
                layers.Conv1D(f, kernel_size, padding="same", activation="relu")
            )
        self.output_conv = layers.Conv1D(1, kernel_size, padding="same", activation="linear")

    def call(self, x):
        x = self.deconv_entry(x)
        x = self.upsample(x)
        for conv in self.conv_layers:
            x = conv(x)
        return self.output_conv(x)

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }


@keras.saving.register_keras_serializable()
class CNNAutoencoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters

        self.encoder = CNNEncoder(filters, kernel_size, latent_filters)
        self.decoder = CNNDecoder(filters, kernel_size, latent_filters)

    def call(self, x):
        return self.decoder(self.encoder(x))

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }

## 3. Configuration Grid

In [4]:
CONFIGS = {
    "cnnae_small_k3_f16": {
        "filters": [32],
        "kernel_size": 3,
        "latent_filters": 16,
        "batch_size": 64,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
    "cnnae_medium_k5_f32": {
        "filters": [64, 32],
        "kernel_size": 5,
        "latent_filters": 32,
        "batch_size": 128,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
    "cnnae_deep_k3_f16": {
        "filters": [64, 32, 16],
        "kernel_size": 3,
        "latent_filters": 16,
        "batch_size": 128,
        "lr": 5e-4,
        "epochs": 100,
        "patience": 10,
    },
    "cnnae_wide_k7_f64": {
        "filters": [128, 64],
        "kernel_size": 7,
        "latent_filters": 64,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
}

## 4. Training Loop

In [ ]:
MODEL_FAMILY = "cnnae"


def reshape_for_cnn(X):
    return X.reshape(-1, X.shape[1], 1) if X.ndim == 2 else X


for cfg_name, cfg in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Training: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)

    model = CNNAutoencoder(
        filters=cfg["filters"],
        kernel_size=cfg["kernel_size"],
        latent_filters=cfg["latent_filters"],
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr"]),
        loss="mse",
    )

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss", patience=cfg["patience"], restore_best_weights=True
    )

    history = model.fit(
        X_train_cnn, X_train_cnn,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_split=0.1,
        shuffle=True,
        callbacks=[early_stop],
    )

    model.summary()

    # --- Reconstruction error & threshold ---
    recon = model.predict(X_train_cnn, batch_size=1024, verbose=0)
    train_errors = nu.reconstruction_mse(X_train_cnn, recon)
    threshold = nu.compute_threshold(train_errors, method="percentile", percentile=97)

    # --- Save ---
    model.save(os.path.join(model_dir, f"{cfg_name}.keras"))

    nu.save_training_artifacts(
        model_dir, history, threshold,
        extra_meta={
            "config": cfg, "input_dim": input_dim,
            "model_type": "cnnae", "input_shape": "(N, 78, 1)",
        },
    )
    nu.plot_error_distribution(
        train_errors, threshold,
        title=f"{cfg_name} — Train Error Distribution",
        save_path=os.path.join(model_dir, "error_dist.png"),
    )

    print(f"  Threshold (97th pctl): {threshold:.6f}")
    print(f"  Mean train MSE:        {train_errors.mean():.6f}")

print("\nAll CNN-AE configs trained and saved.")


  Training: cnnae_small_k3_f16
Epoch 1/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 33s 4ms/step - loss: 0.0021 - val_loss: 2.8832e-06
Epoch 2/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: 5.4552e-06 - val_loss: 1.0635e-06
Epoch 3/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: 3.5503e-06 - val_loss: 1.4680e-06
Epoch 4/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step - loss: 2.7580e-06 - val_loss: 1.5838e-06
Epoch 5/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: 2.2826e-06 - val_loss: 6.4023e-07
Epoch 6/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 41s 3ms/step - loss: 1.9659e-06 - val_loss: 1.0496e-06
Epoch 7/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 41s 3ms/step - loss: 1.7954e-06 - val_loss: 4.4447e-07
Epoch 8/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 41s 3ms/step - loss: 1.5020e-06 - val_loss: 3.7716e-07
Epoch 9/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: 1.5274e-06 - val_loss: 4.6150e-07
Epoch 10/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: 1.4019e-06 - val_loss: 2.

## 5. Quick Evaluation (All Models)

Evaluate every trained CNN-AE config on the attack files and produce a combined comparison.

In [ ]:
all_eval_dfs = []

for cfg_name in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Evaluating: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)
    model_path = os.path.join(model_dir, f"{cfg_name}.keras")
    threshold_path = os.path.join(model_dir, "threshold.json")

    # Load saved model and threshold
    saved_model = keras.models.load_model(model_path)
    with open(threshold_path) as f:
        saved_threshold = json.load(f)["threshold"]

    # Evaluate on attack files
    eval_df = nu.evaluate_on_attack_files(
        model=saved_model,
        scaler=scaler,
        feature_cols=feature_cols,
        threshold=saved_threshold,
        reshape_fn=reshape_for_cnn,
    )
    eval_df.insert(0, "Model", cfg_name)

    # Save per-model quick_eval
    eval_df.to_csv(os.path.join(model_dir, "quick_eval.csv"), index=False)
    print(eval_df.to_string(index=False))

    all_eval_dfs.append(eval_df)

    del saved_model  # free memory

# ── Combined comparison across all models ──
combined_df = pd.concat(all_eval_dfs, ignore_index=True)

summary = combined_df.groupby("Model").agg({
    "Accuracy": "mean", "Precision": "mean", "Recall": "mean",
    "F1": "mean", "AUC": "mean",
}).reset_index().sort_values("F1", ascending=False)
summary.columns = ["Model", "Avg_Accuracy", "Avg_Precision", "Avg_Recall", "Avg_F1", "Avg_AUC"]

family_dir = os.path.join(nu.MODELS_ROOT, MODEL_FAMILY)
combined_df.to_csv(os.path.join(family_dir, "all_models_quick_eval.csv"), index=False)
summary.to_csv(os.path.join(family_dir, "model_comparison_summary.csv"), index=False)

print(f"\n{'='*60}")
print("  MODEL COMPARISON SUMMARY (sorted by Avg F1)")
print(f"{'='*60}")
print(summary.to_string(index=False))
print(f"\nCombined results saved → {os.path.join(family_dir, 'all_models_quick_eval.csv')}")
print(f"Summary saved → {os.path.join(family_dir, 'model_comparison_summary.csv')}")